**SAPIANS** · SCC5819 · MÓDULO 06 — PFI · MOLNAR, CAP. 23

# A importância por permutação do preditor selecionado com treino completo

A seleção global pertence ao módulo da base: primeiro escolhemos e congelamos
um preditor. Depois investigamos como seu desempenho muda quando uma entrada
perde a ligação original com as demais informações e com o desfecho.

[README do módulo](../README.md) · [outline da aula](../lecture/outline.md) ·
[internals desta versão](srag_course_pfi_internals.ipynb) ·
[estudo global](../../00-dataset/notebooks/srag_course_study_walkthrough.ipynb) ·
[Colab](https://colab.research.google.com/github/wbendinelli/interpretable-ml-lectures/blob/main/modules/06-pfi/notebooks/srag_course_pfi_walkthrough.ipynb)

---

**Objetivos.** Ler a PFI da nova versão sem retreinar; interpretar log loss,
diferença, razão e dispersão dos sorteios; distinguir rankings, operações
conjuntas e permutações por subgrupos. Este caderno adicional preserva os
cadernos históricos e não transfere seus resultados para outro preditor.

**Estado da evidência.** Enquanto o agregado público não estiver finalizado,
as células imprimem a pendência e não geram figuras ou estimativas de desempenho.


In [1]:
%pip install -q numpy pandas matplotlib


Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_STATE = 42
REPO = pathlib.Path(os.environ.get("INTERPRETABLE_ML_REPO", pathlib.Path.cwd())).resolve()
while REPO != REPO.parent and not (REPO / "tools/sapians.py").exists():
    REPO = REPO.parent
if not (REPO / "tools/sapians.py").exists():
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True, capture_output=True)
sys.path.insert(0, str(REPO / "tools"))
import sapians as SP

WORK = pathlib.Path(tempfile.mkdtemp(prefix="course-pfi-v2-"))
assert not WORK.is_relative_to(REPO)
os.chdir(WORK)
SP.aplicar()
RECORD = REPO / "modules/00-dataset/gold/course_pfi_v2.json"
RESULT = json.loads(RECORD.read_text()) if RECORD.exists() else None
READY = RESULT is not None and RESULT.get("status") == "complete"
if RESULT is not None and not READY and RESULT.get("status") not in {"pending", "incomplete"}:
    raise ValueError("Estado não reconhecido no registro público da PFI.")
if READY:
    protocol = RESULT["protocol"]
    original = float(RESULT["original"]["logloss"])
    rows = RESULT["rows"]
    assert protocol["version"] == "course-full-logloss-pfi-v2"
    assert original > 0 and np.isfinite(original)
else:
    protocol, original, rows = None, None, []
print("Fonte esperada: modules/00-dataset/gold/course_pfi_v2.json")
print("Estado:", "CONCLUÍDO — resultados agregados disponíveis." if READY else
      "PENDENTE — PFI da nova versão ainda não foi publicada; nenhum resultado presumido.")
print("Este caderno não carrega fichas, modelos ajustados ou previsões individuais.")
print("Nenhum ajuste, seleção de modelo ou cálculo de novas previsões é executado.")


Fonte esperada: modules/00-dataset/gold/course_pfi_v2.json
Estado: PENDENTE — PFI da nova versão ainda não foi publicada; nenhum resultado presumido.
Este caderno não carrega fichas, modelos ajustados ou previsões individuais.
Nenhum ajuste, seleção de modelo ou cálculo de novas previsões é executado.


## §1 — Qual preditor e qual avaliação estamos explicando?

O registro precisa identificar o mesmo artefato escolhido no estudo global,
sua estratégia de treinamento e a avaliação temporal concluída. A seleção
ocorre antes da explicação. Treinamento com pesos ou reamostragem, quando
selecionado, modifica o preditor; permutar uma entrada para explicar seu
desempenho mantém esse preditor fixo.

Aqui lemos somente o agregado produzido pelo executor da PFI. O arquivo contém
perdas por sorteio e metadados, sem fichas ou probabilidades individuais.


In [3]:
def canonical_hash(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()
    return hashlib.sha256(encoded).hexdigest()

def verify_document(document, key):
    expected = document[key]
    assert isinstance(expected, str) and len(expected) == 64
    assert canonical_hash({k: v for k, v in document.items() if k != key}) == expected

if READY:
    verify_document(RESULT, "results_sha256")
    verify_document(protocol, "pfi_protocol_sha256")
    assert protocol["selection_sha256"] == protocol["model"]["selection_sha256"]
    assert protocol["canonical_model"] == protocol["model"]["canonical_model"]
    assert RESULT["integrity"]["fit_performed"] is False
    print("Hashes internos do resultado e do protocolo: conferem.")
    print("Versão da PFI:", protocol["version"])
    print("Versão do preditor:", protocol["model"]["version"])
    print("Família selecionada:", protocol["model"]["family"])
    print("Estratégia selecionada:", protocol["model"]["strategy"])
    print("Identificador do candidato:", protocol["model"]["candidate_id"])
    print("Regra de seleção:", protocol["model"]["selection_rule"])
    print("Ano da avaliação:", str(protocol["period"]))
    print("Registros de avaliação:", SP.pt_int(protocol["n_evaluation"]))
    print("Óbitos observados:", SP.pt_int(protocol["events"]))
    print("Entradas brutas:", SP.pt_int(len(protocol["features"])))
    print("Repetições por operação:", SP.pt_int(protocol["repeats"]))
    print("Semente de permutação:", SP.pt_int(protocol["permutation_seed"]))
    print("Probabilidades recalibradas:", "sim" if protocol["model"]["probabilities_recalibrated"] else "não")
    print("Condição da avaliação: período temporal histórico já observado; análise exploratória.")
else:
    print("PENDENTE — não há preditor nem desempenho da nova PFI a relatar neste registro.")


PENDENTE — não há preditor nem desempenho da nova PFI a relatar neste registro.


**O que olhar.**

- A família e a estratégia impressas identificam o preditor realmente escolhido.
- O período e a quantidade de registros condicionam toda interpretação posterior.
- Ausência do agregado final é pendência, não erro nulo nem confirmação do modelo histórico.


## §2 — O que o log loss mede?

Cada ficha tem um desfecho observado e uma probabilidade de óbito prevista.
O log loss cobra uma penalidade maior quando o modelo atribui pouca
probabilidade ao desfecho que de fato ocorreu. Estar muito confiante em uma
previsão incompatível com o resultado observado recebe uma penalidade alta.

\[
\ell(y,p)=-\{y\ln p+(1-y)\ln(1-p)\},
\qquad E=\frac{1}{n}\sum_{i=1}^{n}\ell(y_i,p_i).
\]

O erro original é a **média dessas penalidades nas fichas avaliadas**. Ele não
é a probabilidade média de óbito, uma taxa de classificações incorretas ou uma
variação do risco clínico. Um valor isolado também não certifica a qualidade
do modelo: a comparação com referências e a calibração pertencem ao estudo global.

A célula mostra o erro original e o erro após embaralhar a entrada de maior
importância média, quando os resultados reais desta versão estiverem disponíveis.


In [4]:
if READY:
    selected = next(row for row in rows if row["feature"] == RESULT["ranking"][0])
    measure = selected["metrics"]["logloss"]
    table = pd.DataFrame([
        {"condição": "Entradas originais", "log loss médio": original, "SD dos sorteios": 0.0},
        {"condição": "Entrada permutada: " + selected["label"],
         "log loss médio": measure["after_mean"], "SD dos sorteios": measure["after_sd"]},
    ])
    print(SP.tabela(table, casas=8))
    print("Mesmos registros, mesmos desfechos e mesmo preditor; somente a entrada é redistribuída.")
    print("O SD do original é nulo nesta repetição: ele não foi reamostrado nem reajustado.")
    fig, ax = plt.subplots(figsize=SP.SLOT)
    ax.bar(["Original", "Após permutar"], [original, measure["after_mean"]],
           yerr=[0.0, measure["after_sd"]], color=[SP.CINZA, SP.AZUL], capsize=4)
    ax.set_ylabel("Log loss médio — menor é melhor")
    SP.titulo(fig, "A permutação muda o erro do mesmo preditor", "§2 · média e SD dos sorteios")
    _ = SP.salvar(fig, "course_pfi_v2_original_permutado")
    plt.show()
else:
    print("PENDENTE — erro original e erros após permutação não disponíveis.")


PENDENTE — erro original e erros após permutação não disponíveis.


**O que olhar.**

- As barras representam penalidades médias, não percentuais de óbito.
- A comparação parte de um único erro original; o erro após a permutação varia entre sorteios.
- A diferença entre barras precisa ser interpretada junto da dispersão exibida.


## §3 — Diferença e razão respondem à mesma comparação em escalas distintas

A diferença mede quanto foi acrescentado ao erro original. A razão informa
quantas vezes o erro após permutar representa o erro original.

\[
\mathrm{PFI}^{\mathrm{dif}}_j=E_{\mathrm{perm},j}-E_{\mathrm{orig}},
\qquad
\mathrm{PFI}^{\mathrm{raz}}_j=\frac{E_{\mathrm{perm},j}}{E_{\mathrm{orig}}}.
\]

A referência sem mudança é zero na diferença e um na razão. Uma permutação
pode melhorar o erro observado: nesse caso a diferença é negativa e a razão
é menor que um. A razão não é uma alteração percentual da probabilidade de óbito.

Como todas as entradas compartilham o mesmo erro original positivo, a troca
de escala preserva a ordem das importâncias médias, salvo empates numéricos.


In [5]:
if READY:
    ranked = sorted(rows, key=lambda row: row["metrics"]["logloss"]["rank"])
    top = ranked[:10]
    comparison = pd.DataFrame([{
        "ordem": row["metrics"]["logloss"]["rank"], "entrada": row["label"],
        "original": original, "após médio": row["metrics"]["logloss"]["after_mean"],
        "diferença média": row["metrics"]["logloss"]["mean"],
        "SD diferença": row["metrics"]["logloss"]["sd"],
        "razão média": row["metrics"]["logloss"]["ratio_mean"],
        "SD razão": row["metrics"]["logloss"]["ratio_sd"],
    } for row in top])
    print(SP.tabela(comparison, casas=8))
    positions = np.arange(len(top))
    fig, axes = plt.subplots(1, 2, figsize=SP.FAIXA)
    for ax, key, spread, reference, title in [
        (axes[0], "mean", "sd", 0.0, "Diferença de log loss"),
        (axes[1], "ratio_mean", "ratio_sd", 1.0, "Razão do log loss"),
    ]:
        means = np.array([row["metrics"]["logloss"][key] for row in top])
        deviations = np.array([row["metrics"]["logloss"][spread] for row in top])
        ax.barh(positions, means-reference, left=reference, xerr=deviations,
                color=SP.AZUL, capsize=3, height=.65)
        ax.axvline(reference, color=SP.CINZA, linestyle="--", linewidth=1)
        ax.set_yticks(positions, [row["label"] for row in top] if ax is axes[0] else [])
        ax.invert_yaxis()
        ax.set_xlabel(title + " · média ± SD")
        low, high = min(reference, float((means-deviations).min())), max(reference, float((means+deviations).max()))
        padding = max((high-low)*.12, .00001)
        ax.set_xlim(low-padding, high+padding)
    SP.titulo(fig, "O mesmo erro original liga diferença e razão", "§3 · entradas de maior importância média")
    _ = SP.salvar(fig, "course_pfi_v2_diferenca_razao")
    plt.show()
else:
    print("PENDENTE — nenhuma diferença, razão ou posição da nova PFI foi calculada neste caderno.")


PENDENTE — nenhuma diferença, razão ou posição da nova PFI foi calculada neste caderno.


**O que olhar.**

- Leia a unidade do eixo e a referência tracejada antes de comparar comprimentos.
- O SD acompanha a escala: dividir a diferença pelo erro original também reescala a dispersão.
- A ordem comum decorre do denominador compartilhado; não são experimentos diferentes.


## §4 — Repetir a permutação descreve a variabilidade dos sorteios

O SD amostral resume a dispersão das importâncias entre embaralhamentos.
Nesta análise os registros e o preditor permanecem fixos: esse SD não é um
intervalo de confiança populacional e não inclui a incerteza de selecionar
ou ajustar outro modelo.

O erro padrão de Monte Carlo da média diminui com mais sorteios. Ele descreve
a precisão numérica da média desses embaralhamentos, não a representatividade
clínica da amostra. A posição de uma entrada também pode variar entre repetições,
especialmente quando importâncias próximas trocam de ordem.


In [6]:
if READY:
    stability = pd.DataFrame([{
        "entrada": row["label"], "aumento médio": row["metrics"]["logloss"]["mean"],
        "SD": row["metrics"]["logloss"]["sd"], "erro MC da média": row["metrics"]["logloss"]["mcse"],
        "posição média entre sorteios": row["metrics"]["logloss"]["rank_mean"],
        "SD da posição": row["metrics"]["logloss"]["rank_sd"],
        "fração entre as primeiras": row["metrics"]["logloss"]["top10_fraction"],
    } for row in top])
    print("Repetições registradas:", SP.pt_int(protocol["repeats"]))
    print("Limite de posições usado na fração:", SP.pt_int(10))
    print("Convenção do SD amostral: ddof =", SP.pt_int(1))
    print(SP.tabela(stability, casas=8))
    delta = np.asarray(selected["metrics"]["logloss"]["delta"])
    mean, sd = float(delta.mean()), float(delta.std(ddof=1))
    fig, ax = plt.subplots(figsize=SP.SLOT)
    ax.scatter(np.arange(1, len(delta)+1), delta, color=SP.AZUL, s=24)
    ax.axhline(mean, color=SP.ESCURO, label="Média dos sorteios")
    ax.axhspan(mean-sd, mean+sd, color=SP.AZUL, alpha=.12, label="Média ± SD; não é IC")
    ax.axhline(0, color=SP.CINZA, linewidth=1, linestyle="--")
    ax.set_xlabel("Repetição")
    ax.set_ylabel("Log loss após − log loss original")
    ax.legend(frameon=False)
    SP.titulo(fig, "Os sorteios variam; o preditor e os casos permanecem fixos", "§4 · " + selected["label"])
    _ = SP.salvar(fig, "course_pfi_v2_repeticoes")
    plt.show()
else:
    print("PENDENTE — dispersão e estabilidade do ranking dependem dos sorteios finalizados.")


PENDENTE — dispersão e estabilidade do ranking dependem dos sorteios finalizados.


**O que olhar.**

- O número de repetições condiciona o resumo e é impresso acima.
- A faixa mostra dispersão entre sorteios, sem prometer cobertura de um parâmetro populacional.
- Ordem pela média e posição média entre sorteios são resumos diferentes.


## §5 — Mover um par junto ou mover uma entrada dentro de subgrupos?

A permutação marginal de cardiopatia rompe suas ligações com as demais
informações. Na operação conjunta, cardiopatia e diabetes recebem a **mesma
linha doadora**: suas combinações observadas viajam juntas.

Na operação por subgrupos, diabetes permanece no receptor. Cardiopatia recebe
um doador do **mesmo estado observado de diabetes**, inclusive o estado
desconhecido. Assim preservamos a composição do par, mas modificamos a pergunta:
qual informação adicional de cardiopatia aparece dentro desses grupos?

Esse agrupamento não representa uma distribuição condicional perfeita dadas
todas as outras entradas. Importância conjunta também não é uma estimativa
isolada de interação, soma obrigatória das importâncias marginais ou efeito causal.


In [7]:
if READY:
    diagnostics = RESULT["dependence"]["results"]
    names = {"cardiopati": "Cardiopatia marginal", "diabetes": "Diabetes marginal",
             "joint": "Par conjunto", "within": "Cardiopatia dentro de diabetes"}
    table = pd.DataFrame([{
        "operação": names[row["key"]], "original": original,
        "após médio": row["metrics"]["logloss"]["after_mean"],
        "aumento médio": row["metrics"]["logloss"]["mean"],
        "SD": row["metrics"]["logloss"]["sd"],
        "razão média": row["metrics"]["logloss"]["ratio_mean"],
    } for row in diagnostics])
    print(SP.tabela(table, casas=8))
    print("Grupos de diabetes e quantidades registradas pelo executor:")
    print(SP.tabela(pd.DataFrame([{"estado": key, "registros": value}
                                  for key, value in protocol["dependence"]["group_counts"].items()])))
    print("O executor declara auditoria das contagens do par em todas as operações conjuntas e por subgrupos.")
    fig, ax = plt.subplots(figsize=SP.SLOT)
    positions = np.arange(len(diagnostics))
    means = [row["metrics"]["logloss"]["mean"] for row in diagnostics]
    deviations = [row["metrics"]["logloss"]["sd"] for row in diagnostics]
    ax.barh(positions, means, xerr=deviations, color=SP.AZUL, capsize=3, height=.65)
    ax.set_yticks(positions, [names[row["key"]] for row in diagnostics])
    ax.invert_yaxis()
    ax.axvline(0, color=SP.CINZA, linestyle="--", linewidth=1)
    ax.set_xlabel("Log loss após − log loss original · média ± SD")
    SP.titulo(fig, "Mudar a informação preservada muda a pergunta", "§5 · mesmo teste e preditor")
    _ = SP.salvar(fig, "course_pfi_v2_dependencia")
    plt.show()
else:
    print("PENDENTE — não há resultado novo de permutação conjunta ou por subgrupos.")


PENDENTE — não há resultado novo de permutação conjunta ou por subgrupos.


**O que olhar.**

- As operações individuais e a conjunta preservam informações diferentes.
- Dentro de diabetes, a coluna que define os grupos permanece fixa.
- Compare magnitude e SD sem chamar a diferença entre operações de causalidade ou interação isolada.


## §6 — O que o livro diz e o que este registro demonstra

Molnar descreve a PFI como mudança no desempenho de um modelo fixo após
permutar uma entrada. Sua interpretação depende do modelo, dos dados, da
métrica e do mecanismo de perturbação. O livro discute a dificuldade trazida
por dependências entre entradas e a alternativa de trabalhar em subgrupos.

Neste estudo, a origem e a seleção do preditor são registradas separadamente.
A avaliação histórica já observada continua exploratória. O agregado permite
conferir as estatísticas das perdas publicadas; sozinho, não permite refazer
as previsões, conferir cada ficha ou demonstrar todas as invariantes do transporte.
O [internals](srag_course_pfi_internals.ipynb) torna essa fronteira explícita.


In [8]:
if READY:
    print("Arquivo agregado íntegro:", "sim")
    print("Entradas com perdas por sorteio:", SP.pt_int(len(rows)))
    print("Modelo ajustado ou modificado neste caderno:", "não")
    print("Previsões individuais reproduzidas neste caderno:", "não")
    print("Próxima exigência para a apresentação: sincronizar texto, figuras e metadados com esta versão do preditor.")
else:
    print("PENDENTE — aguardar seleção global, avaliação do vencedor e execução integral da PFI.")
    print("Sem agregado completo, nenhuma conclusão empírica é transferida do estudo histórico.")


PENDENTE — aguardar seleção global, avaliação do vencedor e execução integral da PFI.
Sem agregado completo, nenhuma conclusão empírica é transferida do estudo histórico.


**O que olhar.**

- Integridade de um agregado e reprodução das previsões são verificações distintas.
- O ranking resume perda preditiva; ele não determina causa, direção do risco ou exclusão automática de uma entrada.
- Uma nova referência exige novas explicações e sincronização dos materiais.

## Fechamento — primeiro o preditor, depois sua dependência de informação

Mantemos a sequência: selecionar o preditor, congelar seu artefato, avaliar
seu desempenho e explicar esse mesmo objeto. Permutar a entrada modifica
seu vínculo com as fichas, sem criar um novo treinamento. Diferença, razão,
dispersão dos sorteios e mecanismo de transporte precisam acompanhar a conclusão.

**Referências.** Molnar, *Interpretable Machine Learning*, [Permutation Feature Importance](https://christophm.github.io/interpretable-ml-book/feature-importance.html).
Scikit-learn, [log loss](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html) e
[permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html).
